# 🎵 Music Recommendation System (ALS Training Test)

Notebook này được dùng để thử nghiệm thuật toán **ALS (Collaborative Filtering)** trên tập dữ liệu nghe nhạc từ tầng Silver. Chúng ta sẽ chuyển đổi dữ liệu thô thành định dạng (User, Item, Rating) để huấn luyện mô hình gợi ý.

### 1. Khởi tạo Spark Session

In [22]:
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline
from pyspark.sql.functions import col, count, explode, collect_list

spark = SparkSession.builder \
    .appName("RecommendationTest") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session đã sẵn sàng!")

Spark Session đã sẵn sàng!


### 2. Load dữ liệu Silver từ HDFS

In [23]:
silver_path = "hdfs://master:9000/datalake/silver/eventsim/"
df = spark.read.parquet(silver_path)

print(f"Tổng số bản ghi: {df.count()}")
df.select("userId", "song", "artist").show(5)

Tổng số bản ghi: 714876
+------+--------------------+-------------------+
|userId|                song|             artist|
+------+--------------------+-------------------+
|    10|    Ain't Misbehavin|          Sam Cooke|
|    10|            The Gift|Angels and Airwaves|
|    10|            Everlong|       Foo Fighters|
|    10|           Attention|     The Raconteurs|
|    10|Alive (Ron Reeser...|       Annet Artani|
+------+--------------------+-------------------+
only showing top 5 rows



### 3. Chuẩn bị dữ liệu (Data Preparation)

In [24]:
# Filter dữ liệu cần thiết
raw_interactions = df.filter(col("userId").isNotNull() & col("song").isNotNull())

# Tính play count
interactions_df = raw_interactions.groupBy("userId", "song").agg(count("*").alias("play_count"))

# Ánh xạ String -> Integer
user_indexer = StringIndexer(inputCol="userId", outputCol="user_idx").setHandleInvalid("skip")
song_indexer = StringIndexer(inputCol="song", outputCol="song_idx").setHandleInvalid("skip")

indexer_pipeline = Pipeline(stages=[user_indexer, song_indexer])
pipeline_model = indexer_pipeline.fit(interactions_df)
final_df = pipeline_model.transform(interactions_df)

final_df.show(5)

+------+--------------------+----------+--------+--------+
|userId|                song|play_count|user_idx|song_idx|
+------+--------------------+----------+--------+--------+
|   116|                Undo|        12|   433.0|     1.0|
|   133|       Bleeding Love|        12|    22.0|   787.0|
|   137|A Lack of Common ...|        12|     3.0|  2658.0|
|   137|           Year 3000|        12|     3.0|  1763.0|
|   142|LoveStoned/I Thin...|        12|   507.0|  1542.0|
+------+--------------------+----------+--------+--------+
only showing top 5 rows



### 4. Huấn luyện mô hình ALS

In [25]:
als = ALS(maxIter=10, regParam=0.1, userCol="user_idx", itemCol="song_idx", ratingCol="play_count", 
          coldStartStrategy="drop", nonnegative=True)

model = als.fit(final_df)
print("Model training hoàn tất!")

Model training hoàn tất!


### 5. Giao diện Gợi ý trực quan (Visual Recommendations)
Thay vì xem bảng thô, chúng ta sẽ in ra kết quả theo dạng Dashboard để dễ theo dõi.

In [26]:
# 1. Lấy gợi ý thô
user_recs = model.recommendForAllUsers(5)

# 2. Join và gom nhóm bài hát theo từng User
flat_recs = user_recs.withColumn("recommendation", explode(col("recommendations"))) \
                     .select("user_idx", "recommendation.song_idx")

song_mapping = final_df.select("song", "song_idx").distinct()
user_mapping = final_df.select("userId", "user_idx").distinct()

display_df = flat_recs.join(song_mapping, "song_idx") \
                      .join(user_mapping, "user_idx") \
                      .groupBy("userId") \
                      .agg(collect_list("song").alias("recommended_songs"))

# 3. In ra giao diện Dashboard đẹp mắt
print("="*60)
print("🌟 MUSIC RECOMMENDATION DASHBOARD 🌟")
print("="*60)

results = display_df.limit(10).collect() # Lấy 10 user mẫu để demo

for row in results:
    print(f"\n👤 User ID: {row['userId']}")
    for i, song in enumerate(row['recommended_songs'], 1):
        print(f"  {i}. 🎵 {song}")
print("\n" + "="*60)

🌟 MUSIC RECOMMENDATION DASHBOARD 🌟

👤 User ID: 10
  1. 🎵 Dumb
  2. 🎵 What's It Gonna Be
  3. 🎵 Beautiful World (From Colin Hay album "Going Somewhere" HEAD038)
  4. 🎵 Let's Try It Again
  5. 🎵 Mixer At Delta Chi

👤 User ID: 100
  1. 🎵 Dumb
  2. 🎵 Take Me Out
  3. 🎵 You're The One
  4. 🎵 Beautiful World (From Colin Hay album "Going Somewhere" HEAD038)
  5. 🎵 Let's Try It Again

👤 User ID: 1000
  1. 🎵 What's It Gonna Be
  2. 🎵 Charlotte Sometimes
  3. 🎵 Plus Rien Ne M'Etonne
  4. 🎵 Let's Try It Again
  5. 🎵 Look What You've Done (Album Version)

👤 User ID: 1001
  1. 🎵 Charlotte Sometimes
  2. 🎵 You're The One
  3. 🎵 Beautiful World (From Colin Hay album "Going Somewhere" HEAD038)
  4. 🎵 Let's Try It Again
  5. 🎵 Mixer At Delta Chi

👤 User ID: 101
  1. 🎵 Clint Eastwood
  2. 🎵 Undo
  3. 🎵 Greece 2000
  4. 🎵 World News
  5. 🎵 Every Little Thing She Does Is Magic

👤 User ID: 102
  1. 🎵 City Love
  2. 🎵 Undo
  3. 🎵 Talamak
  4. 🎵 Beautiful World (From Colin Hay album "Going Somewhere" HEAD038